# ❄️ 妙高市 降雪・積雪データ可視化 — コード解説ノートブック

このノートブックでは、妙高市のウェブサイトから雪のデータを取得して可視化する Python プログラムを、  
**ステップごとにデータの変化を確認しながら** 学習します。

## 扱う技術スタック

| ライブラリ | 役割 |
|---|---|
| `requests` | ウェブページの HTML を取得する |
| `BeautifulSoup` | HTML を解析してテーブルを抽出する |
| `pandas` | データを表形式（DataFrame）に整形・加工する |
| `plotly` | インタラクティブなグラフを描画する |

## データの流れ（全体像）

```
ウェブサイト (HTML)
    ↓  requests でダウンロード
HTML 文字列
    ↓  BeautifulSoup でテーブル抽出
HTML テーブル（行・列の構造）
    ↓  ループで各セルを読み取り
Python リスト（辞書のリスト）
    ↓  pandas DataFrame に変換
整形済み DataFrame
    ↓  plotly でグラフ化
インタラクティブグラフ
```

## 0. ライブラリのインストール

Google Colab には `pandas` や `requests` は最初から入っていますが、  
`lxml`（HTML パーサー）は念のためインストールします。

In [ ]:
!pip install lxml beautifulsoup4 plotly --quiet
print("インストール完了")

In [ ]:
# 使用するライブラリをすべてインポート
import requests
from bs4 import BeautifulSoup
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from typing import Optional

print("requests:", requests.__version__)
print("pandas:", pd.__version__)

---
## 1. ステップ1: ウェブページの HTML を取得する

### requests とは？

`requests` は Python でもっとも広く使われる HTTP クライアントライブラリです。  
ブラウザの代わりに Python からウェブページを取得（GET リクエスト）できます。

```
あなたのPC                妙高市ウェブサーバー
    |--- GET /docs/64741.html --->|
    |<---------- HTML ---------- |
```

今回は 2025年1月のデータページを例に使います。

In [ ]:
# 2025年1月のデータ URL（data_urls.csv に登録されているもの）
URL = "https://www.city.myoko.niigata.jp/docs/64741.html"
YEAR = 2025
MONTH = 1

# HTTP GET リクエストを送信
response = requests.get(URL, timeout=10)

# ステータスコードを確認（200 = 成功）
print("ステータスコード:", response.status_code)
print("コンテンツタイプ:", response.headers.get('Content-Type', '不明'))

In [ ]:
# 文字コードを自動検出して設定（日本語サイトに重要！）
# apparent_encoding は chardet ライブラリで推定した文字コードを返す
print("推定された文字コード:", response.apparent_encoding)
response.encoding = response.apparent_encoding

# 取得した HTML の冒頭を確認
html_text = response.text
print(f"\nHTMLサイズ: {len(html_text):,} 文字")
print("\n--- HTML 冒頭 500文字 ---")
print(html_text[:500])

### ポイント: `response.raise_for_status()`

本番コードでは `response.raise_for_status()` を呼ぶことで、  
404（ページなし）や 500（サーバーエラー）などのエラーを例外として受け取れます。

In [ ]:
# エラーチェックの例
try:
    response.raise_for_status()  # 2xx 以外なら HTTPError を raise する
    print("✅ リクエスト成功！")
except requests.HTTPError as e:
    print(f"❌ HTTPエラー: {e}")

---
## 2. ステップ2: BeautifulSoup で HTML を解析する

### BeautifulSoup とは？

HTML は入れ子構造（ツリー構造）を持つテキストです。  
`BeautifulSoup` はこのツリーを Python のオブジェクトとして扱えるようにしてくれます。

```html
<html>
  <body>
    <table>          ← find_all("table") で全テーブルを取得
      <tr>           ← find_all("tr") で全行を取得
        <td>3日</td>  ← get_text() でテキストを取得
        <td>10</td>
      </tr>
    </table>
  </body>
</html>
```

In [ ]:
# BeautifulSoup オブジェクトを作成
# "lxml" は HTML の解析エンジン（lxml は高速で正確）
soup = BeautifulSoup(html_text, "lxml")

print("BeautifulSoup オブジェクトの型:", type(soup))
print("ページタイトル:", soup.title.get_text() if soup.title else "タイトルなし")

In [ ]:
# ページ内のすべてのテーブルを取得
all_tables = soup.find_all("table")
print(f"ページ内のテーブル数: {len(all_tables)} 個")

# 各テーブルの概要を確認
for i, tbl in enumerate(all_tables):
    rows = tbl.find_all("tr")
    # テーブル内テキストの先頭50文字
    preview = tbl.get_text(" ", strip=True)[:80]
    print(f"  テーブル {i}: {len(rows)}行 | 内容: {preview}...")

### テーブル選択ロジック

ページには複数のテーブルが含まれている場合があります。  
プログラムでは「3つの観測地点名がすべて含まれるテーブル」を優先的に選択します。

In [ ]:
# 観測地点名（main.py の LOCATIONS 定数と同じ）
LOCATIONS = ["新井消防署", "頸南消防署", "妙高市役所 妙高支所"]

def pick_data_table(soup: BeautifulSoup) -> Optional[BeautifulSoup]:
    """観測所名が含まれるテーブルを優先して返す。なければ最初のテーブル。"""
    tables = soup.find_all("table")
    if not tables:
        return None

    for tbl in tables:
        text = tbl.get_text(" ", strip=True)
        # all() は「リスト内の全要素が True のとき True」
        if all(loc in text for loc in LOCATIONS):
            print(f"✅ 観測所名を含むテーブルを発見！")
            return tbl

    print("⚠️ 観測所名のテーブルなし → 最初のテーブルを使用")
    return tables[0]

# 実際に選択されたテーブルを取得
data_table = pick_data_table(soup)
print(f"選択されたテーブルの行数: {len(data_table.find_all('tr'))} 行")

In [ ]:
# テーブルの生の HTML を確認（最初の3行だけ）
rows = data_table.find_all("tr")
print(f"テーブル全行数: {len(rows)} 行\n")

print("=== ヘッダー行 (rows[0]) の HTML ===")
print(rows[0].prettify()[:500])

In [ ]:
# ヘッダー行のテキストを取り出して列構造を把握する
print("=== 各行のテキスト内容（最初の5行）===")
for i, row in enumerate(rows[:5]):
    cols = row.find_all(["td", "th"])  # td（データセル）と th（ヘッダーセル）の両方
    texts = [c.get_text(strip=True) for c in cols]
    print(f"行 {i} ({len(texts)}列): {texts}")

### テーブルの列構造

ヘッダーを確認すると、以下の構造になっています:

| 列インデックス | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| 内容 | 日 | 降雪1 | 積雪1 | 降雪2 | 積雪2 | 降雪3 | 積雪3 |
| 対応地点 | - | 新井消防署 | 新井消防署 | 頸南消防署 | 頸南消防署 | 妙高市役所 | 妙高市役所 |

地点 `i`（0始まり）の列インデックス:
- 降雪量: `i * 2 + 1`
- 積雪量: `i * 2 + 2`

---
## 3. ステップ3: テーブルデータを Python リストに変換する

HTML テーブルの各行をループし、辞書（dict）形式でデータを収集します。

In [ ]:
def to_float_or_none(s: str) -> Optional[float]:
    """
    文字列を float に変換するヘルパー関数。
    "-", "--", "" など「値なし」を表す文字列は None を返す。
    "30-" のような末尾の "-" は除去して変換する。
    """
    x = (s or "").strip()
    if x in {"-", "--", ""}:  # 欠損値パターン
        return None
    x = x.rstrip("-").strip()  # "30-" → "30"
    if x == "":
        return None
    try:
        return float(x)
    except ValueError:
        return None

# 動作確認
test_cases = ["15", "30.5", "-", "--", "", "30-", "abc"]
for t in test_cases:
    result = to_float_or_none(t)
    print(f"  '{t}' → {result}")

In [ ]:
# テーブルの全行をループしてデータを収集
data_rows = []

for row in rows[1:]:  # rows[0] はヘッダー行なのでスキップ
    cols = row.find_all(["td", "th"])

    # 列数が少なすぎる行はスキップ（ヘッダー行や区切り行など）
    if len(cols) < 7:
        continue

    cols_text = [c.get_text(strip=True) for c in cols]

    # 1列目から日付を取る: "3日" → "3" → 3
    day_text = cols_text[0].replace("日", "").strip()
    if not day_text.isdigit():  # 数字でなければスキップ（ヘッダー文字列など）
        continue
    day = int(day_text)

    # 3つの観測地点に対してループ
    for i, location in enumerate(LOCATIONS):
        snowfall_idx = i * 2 + 1   # 降雪量の列インデックス
        snowdepth_idx = i * 2 + 2  # 積雪量の列インデックス

        snowfall_raw = cols_text[snowfall_idx] if snowfall_idx < len(cols_text) else "-"
        snowdepth_raw = cols_text[snowdepth_idx] if snowdepth_idx < len(cols_text) else "-"

        snowfall_cm = to_float_or_none(snowfall_raw)
        snowdepth_cm = to_float_or_none(snowdepth_raw)

        # 積雪量が負になるのは異常値なので None に
        if snowdepth_cm is not None and snowdepth_cm < 0:
            snowdepth_cm = None

        data_rows.append({
            "year": YEAR,
            "month": MONTH,
            "day": day,
            "location": location,
            "snowfall_cm": snowfall_cm,
            "snowdepth_cm": snowdepth_cm,
        })

print(f"抽出されたデータ行数: {len(data_rows)} 件")
print("\n最初の3件:")
for row in data_rows[:3]:
    print(" ", row)

---
## 4. ステップ4: pandas DataFrame に変換する

### DataFrame とは？

辞書のリストを `pd.DataFrame()` に渡すと、Excel のような表形式データに変換されます。

```python
# 辞書のリスト
[{"day": 1, "location": "新井消防署", "snowfall_cm": None},
 {"day": 1, "location": "頸南消防署", "snowfall_cm": 5.0}, ...]

#     ↓ pd.DataFrame() で変換

#  day  location        snowfall_cm
#  1    新井消防署       NaN
#  1    頸南消防署       5.0
```

In [ ]:
# リストから DataFrame を作成
df = pd.DataFrame(data_rows)

print("DataFrame の形状:", df.shape, "（行数, 列数）")
print("\n列名と型:")
print(df.dtypes)
print("\n先頭 9 行:")
df.head(9)

In [ ]:
# データの基本統計量を確認
print("=== 基本統計量 ===")
df[["snowfall_cm", "snowdepth_cm"]].describe()

In [ ]:
# 欠損値（NaN）の確認
print("=== 欠損値の数 ===")
print(df.isnull().sum())
print(f"\n全体の欠損率: {df.isnull().mean().mean():.1%}")

### Tidy データ（整然データ）とは？

このプログラムでは、生のテーブル（ワイド形式）を**Tidy データ**（ロング形式）に変換しています。

**元のテーブル（ワイド形式）:**
| 日 | 新井_降雪 | 新井_積雪 | 頸南_降雪 | 頸南_積雪 | ... |
|---|---|---|---|---|---|
| 1 | - | - | - | - | ... |
| 2 | 5 | 10 | 3 | 8 | ... |

**Tidy 形式（ロング形式）:**
| day | location | snowfall_cm | snowdepth_cm |
|---|---|---|---|
| 1 | 新井消防署 | NaN | NaN |
| 1 | 頸南消防署 | NaN | NaN |
| 2 | 新井消防署 | 5.0 | 10.0 |
| 2 | 頸南消防署 | 3.0 | 8.0 |

Tidy 形式にすると `df[df["location"] == "新井消防署"]` のようなフィルタリングが簡単になります。

In [ ]:
# 観測地点でフィルタリングする例
location = "新井消防署"
filtered = df[
    (df["year"] == YEAR) &
    (df["month"] == MONTH) &
    (df["location"] == location)
].sort_values("day")

print(f"{YEAR}年{MONTH}月 / {location} のデータ ({len(filtered)}件):")
filtered.head(10)

In [ ]:
# 地点別の月合計・最大値を集計
print("=== 地点別 月間集計 ===")
summary = df.groupby("location")[["snowfall_cm", "snowdepth_cm"]].agg(
    snowfall_total=("snowfall_cm", "sum"),
    snowfall_max=("snowfall_cm", "max"),
    snowdepth_max=("snowdepth_cm", "max"),
)
summary

---
## 5. ステップ5: Plotly でインタラクティブグラフを作成する

### グラフの設計

- **積雪量（棒グラフ、左軸）**: 地面に積もった雪の深さ（cm）
- **降雪量（折れ線グラフ、右軸）**: 前日から当日朝9時までに降った雪の量（cm）

2つの量はスケールが異なるため（積雪は最大300cm、降雪は最大100cm程度）、  
**secondary_y（第2軸）** を使って別々のスケールで表示します。

In [ ]:
# まず secondary_y を使わずに普通のグラフを作って問題を確認
location = "新井消防署"
plot_df = df[
    (df["year"] == YEAR) &
    (df["month"] == MONTH) &
    (df["location"] == location)
].sort_values("day")

fig_single = go.Figure()
fig_single.add_trace(go.Bar(
    x=plot_df["day"],
    y=plot_df["snowdepth_cm"],
    name="積雪量",
    opacity=0.7
))
fig_single.add_trace(go.Scatter(
    x=plot_df["day"],
    y=plot_df["snowfall_cm"],
    name="降雪量",
    mode="lines+markers",
    line=dict(color="red"),
))
fig_single.update_layout(
    title=f"単一Y軸（問題あり）: {YEAR}年{MONTH}月 / {location}",
    height=350
)
fig_single.show()
print("→ 降雪量（赤線）が積雪量に比べて小さすぎて見えにくい問題がある")

In [ ]:
# make_subplots で secondary_y（第2Y軸）を持つ図を作成
fig_dual = make_subplots(specs=[[{"secondary_y": True}]])

# 積雪量（棒グラフ）を左軸（primary, secondary_y=False）に追加
fig_dual.add_trace(
    go.Bar(
        x=plot_df["day"],
        y=plot_df["snowdepth_cm"],
        name="積雪量",
        opacity=0.7,
    ),
    secondary_y=False,  # 左のY軸
)

# 降雪量（折れ線グラフ）を右軸（secondary_y=True）に追加
fig_dual.add_trace(
    go.Scatter(
        x=plot_df["day"],
        y=plot_df["snowfall_cm"],
        name="降雪量",
        mode="lines+markers",
        line=dict(color="red"),
        marker=dict(color="red"),
    ),
    secondary_y=True,   # 右のY軸
)

# レイアウトの設定
fig_dual.update_layout(
    title=f"{YEAR}年{MONTH}月 / {location}",
    xaxis_title="日",
    hovermode="x unified",   # マウスオーバーで両データを同時表示
    height=400,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

# X軸の範囲を1〜31日に固定
fig_dual.update_xaxes(range=[0.5, 31.5], dtick=5)

# 左Y軸（積雪量）の設定
fig_dual.update_yaxes(
    title_text="積雪量 (cm)",
    range=[0, 300],
    dtick=60,
    secondary_y=False,
)

# 右Y軸（降雪量）の設定
fig_dual.update_yaxes(
    title_text="降雪量 (cm)",
    range=[0, 100],
    dtick=20,
    secondary_y=True,
)

fig_dual.show()

---
## 6. 全体を関数にまとめる

ここまでのステップを関数化すると、`main.py` と同等のコードになります。

In [ ]:
def fetch_snow_data(url: str, year: int, month: int) -> Optional[pd.DataFrame]:
    """指定URLから降雪・積雪データを取得して tidy DataFrame を返す（失敗時 None）"""
    try:
        # --- ステップ1: HTTP リクエスト ---
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = response.apparent_encoding

        # --- ステップ2: BeautifulSoup でテーブル抽出 ---
        soup = BeautifulSoup(response.text, "lxml")
        table = pick_data_table(soup)
        if table is None:
            return None

        rows = table.find_all("tr")
        if len(rows) < 3:
            return None

        # --- ステップ3: 行ループでデータ収集 ---
        data_rows = []
        for row in rows[1:]:
            cols = row.find_all(["td", "th"])
            if len(cols) < 7:
                continue

            cols_text = [c.get_text(strip=True) for c in cols]
            day_text = cols_text[0].replace("日", "").strip()
            if not day_text.isdigit():
                continue
            day = int(day_text)

            for i, location in enumerate(LOCATIONS):
                snowfall_raw = cols_text[i * 2 + 1] if i * 2 + 1 < len(cols_text) else "-"
                snowdepth_raw = cols_text[i * 2 + 2] if i * 2 + 2 < len(cols_text) else "-"
                snowfall_cm = to_float_or_none(snowfall_raw)
                snowdepth_cm = to_float_or_none(snowdepth_raw)
                if snowdepth_cm is not None and snowdepth_cm < 0:
                    snowdepth_cm = None

                data_rows.append({
                    "year": year, "month": month, "day": day,
                    "location": location,
                    "snowfall_cm": snowfall_cm, "snowdepth_cm": snowdepth_cm,
                })

        # --- ステップ4: DataFrame 変換 ---
        if not data_rows:
            return None
        return pd.DataFrame(data_rows)

    except requests.RequestException as e:
        print(f"HTTPエラー: {url} - {e}")
        return None


def create_snow_graph(df: pd.DataFrame, year: int, month: int, location: str) -> go.Figure:
    """降雪・積雪のグラフを作成"""
    filtered_df = df[
        (df["year"] == year) & (df["month"] == month) & (df["location"] == location)
    ].sort_values("day")

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Bar(x=filtered_df["day"], y=filtered_df["snowdepth_cm"], name="積雪量", opacity=0.7),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=filtered_df["day"], y=filtered_df["snowfall_cm"], name="降雪量",
            mode="lines+markers", line=dict(color="red"), marker=dict(color="red"),
        ),
        secondary_y=True,
    )
    fig.update_layout(
        title=f"{year}年{month}月 / {location}", xaxis_title="日",
        hovermode="x unified", height=400, showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.update_xaxes(range=[0.5, 31.5], dtick=5)
    fig.update_yaxes(title_text="積雪量 (cm)", range=[0, 300], dtick=60, secondary_y=False)
    fig.update_yaxes(title_text="降雪量 (cm)", range=[0, 100], dtick=20, secondary_y=True)
    return fig


print("関数を定義しました。次のセルで実行します。")

---
## 7. 複数月・複数地点の比較

data_urls.csv に記載されている URL を使って、異なる月や地点を比較してみましょう。

In [ ]:
# 比較したい月と URL を辞書形式で定義
# （実際のアプリでは data_urls.csv から読み込んでいます）
targets = [
    {"year": 2025, "month": 1, "url": "https://www.city.myoko.niigata.jp/docs/64741.html"},
    {"year": 2025, "month": 2, "url": "https://www.city.myoko.niigata.jp/docs/64742.html"},
]
compare_location = "新井消防署"

for target in targets:
    df_month = fetch_snow_data(target["url"], target["year"], target["month"])
    if df_month is None:
        print(f"⚠️ {target['year']}年{target['month']}月 のデータ取得失敗")
        continue

    print(f"\n✅ {target['year']}年{target['month']}月 データ取得成功 ({len(df_month)}件)")
    fig = create_snow_graph(df_month, target["year"], target["month"], compare_location)
    fig.show()

---
## 8. (発展) 年をまたいだ積雪ピーク値の比較

複数年のデータから月最大積雪量を比較してみます。

In [ ]:
# 複数年の1月データを取得して比較
january_urls = [
    {"year": 2022, "month": 1, "url": "https://www.city.myoko.niigata.jp/docs/30540.html"},
    {"year": 2023, "month": 1, "url": "https://www.city.myoko.niigata.jp/docs/39647.html"},
    {"year": 2024, "month": 1, "url": "https://www.city.myoko.niigata.jp/docs/52957.html"},
    {"year": 2025, "month": 1, "url": "https://www.city.myoko.niigata.jp/docs/64741.html"},
]

all_dfs = []
for item in january_urls:
    d = fetch_snow_data(item["url"], item["year"], item["month"])
    if d is not None:
        all_dfs.append(d)
        print(f"  {item['year']}年{item['month']}月: {len(d)}件取得")

if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n合計データ件数: {len(combined_df)}件")

In [ ]:
if all_dfs:
    # 地点・年ごとの月最大積雪量を集計
    peak = combined_df.groupby(["year", "month", "location"])["snowdepth_cm"].max().reset_index()
    peak.columns = ["year", "month", "location", "max_snowdepth_cm"]
    peak["label"] = peak["year"].astype(str) + "年" + peak["month"].astype(str) + "月"

    # 棒グラフで比較
    fig_compare = go.Figure()
    for loc in LOCATIONS:
        loc_data = peak[peak["location"] == loc]
        fig_compare.add_trace(go.Bar(
            x=loc_data["label"],
            y=loc_data["max_snowdepth_cm"],
            name=loc,
            opacity=0.85,
        ))

    fig_compare.update_layout(
        title="1月 月間最大積雪量（年別・地点別）",
        xaxis_title="年月",
        yaxis_title="最大積雪量 (cm)",
        barmode="group",
        height=400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig_compare.show()

---
## まとめ

このノートブックで学んだこと:

| ステップ | 技術 | 概要 |
|---|---|---|
| 1 | `requests.get()` | ウェブページの HTML を取得 |
| 2 | `BeautifulSoup` | HTML を解析してテーブルを特定 |
| 3 | Python ループ + `str` 操作 | テーブルの各セルをデータに変換 |
| 4 | `pd.DataFrame()` | リストを表形式データに変換・集計 |
| 5 | `plotly` | 2軸グラフで可視化 |

### 実際のアプリ（main.py）での追加機能

- **Streamlit**: Python だけでウェブUIを構築（`st.selectbox` でパラメータ選択）
- **キャッシュ**: `@st.cache_data(ttl=3600)` で同じURLの再取得を1時間スキップ
- **履歴保存**: 取得済みデータを CSV に保存してウェブアクセスを最小化
- **エラーハンドリング**: `try/except` で取得失敗時もアプリがクラッシュしないよう保護